# Figure 9.21: Complexity, Bias, Variance, and Error

Training error falls with complexity, while test error and variance eventually turn upward.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def rng_from_seed(seed):
    return np.random.default_rng(int(seed))


def linspace(a, b, n):
    return np.linspace(a, b, int(n))


def base_curve(x):
    return 1.0 + 0.7 * x - 0.35 * x**2 + 0.06 * x**3


def poly_design(x, degree):
    return np.vstack([x**k for k in range(degree + 1)]).T


def ridge_fit(x, y, degree, lam):
    X = poly_design(x, degree)
    I = np.eye(degree + 1)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def poly_predict(theta, x):
    X = poly_design(np.asarray(x), len(theta) - 1)
    return X @ theta


def line_fit(x, y, lam=0.0):
    X = np.column_stack([np.ones_like(x), x])
    I = np.eye(2)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def line_predict(theta, x):
    x = np.asarray(x)
    return theta[0] + theta[1] * x


def poly_data(seed=0, n=40, obs_noise=0.3, gt_noise=0.0):
    rng = rng_from_seed(seed)
    x = np.linspace(-3, 3, n)
    y_true = base_curve(x)
    y_star = y_true + gt_noise * rng.normal(size=n)
    y = y_star + obs_noise * rng.normal(size=n)
    return x, y, y_true, y_star

def summarize(noise, max_degree, trials, seed):
    train_err, test_err, bias2, variance = [], [], [], []
    test_x = np.linspace(-3, 3, 80)
    truth = base_curve(test_x)
    for degree in range(1, max_degree + 1):
        preds = []
        tr = 0.0
        te = 0.0
        for t in range(trials):
            x, y, _, _ = poly_data(seed=seed * 100 + t + 1, n=36, obs_noise=noise, gt_noise=0.0)
            th = ridge_fit(x, y, degree, 0.0)
            pred_train = poly_predict(th, x)
            tr += np.mean((pred_train - y) ** 2)
            pred_test = poly_predict(th, test_x)
            te += np.mean((pred_test - truth) ** 2)
            preds.append(pred_test)
        preds = np.stack(preds, axis=0)
        mean_pred = preds.mean(axis=0)
        train_err.append(tr / trials)
        test_err.append(te / trials)
        bias2.append(np.mean((mean_pred - truth) ** 2))
        variance.append(np.mean(np.var(preds, axis=0)))
    return np.arange(1, max_degree + 1), np.array(train_err), np.array(test_err), np.array(bias2), np.array(variance)


def draw(noise=0.25, max_degree=12, trials=24, seed=5):
    x, train_err, test_err, bias2, variance = summarize(noise, max_degree, trials, seed)
    fig, axs = plt.subplots(1, 2, figsize=(13.0, 4.4))
    axs[0].plot(x, train_err, "-o", color="#2563eb", label="train error")
    axs[0].plot(x, test_err, "-o", color="#dc2626", label="test error")
    axs[0].set_title("Training vs test error")
    axs[0].set_xlabel("model degree")
    axs[0].set_ylabel("MSE")
    axs[0].grid(alpha=0.2)
    axs[0].legend()
    axs[1].plot(x, bias2, "-o", color="#059669", label=r"bias$^2$")
    axs[1].plot(x, variance, "-o", color="#d97706", label="variance")
    axs[1].plot(x, bias2 + variance, ":o", color="#7c3aed", label=r"bias$^2$ + variance")
    axs[1].set_title("Bias-variance view")
    axs[1].set_xlabel("model degree")
    axs[1].grid(alpha=0.2)
    axs[1].legend()
    plt.show()

widgets.interact(
    draw,
    noise=widgets.FloatSlider(min=0.05, max=0.8, step=0.05, value=0.25, continuous_update=False),
    max_degree=widgets.IntSlider(min=4, max=14, step=1, value=12, continuous_update=False),
    trials=widgets.IntSlider(min=10, max=60, step=2, value=24, continuous_update=False),
    seed=widgets.IntSlider(min=1, max=20, step=1, value=5, continuous_update=False),
)
